# Bgee — Gene Expression Evolution Database

**Bgee** (Gene Expression Evolution) is an ELIXIR Core Data Resource for the retrieval and comparison of gene expression patterns across multiple animal species. It integrates heterogeneous expression data types — curated and quality-controlled — and maps them to standardised anatomical and developmental ontologies.

Key data types stored in Bgee:
| Data type | Description |
|---|---|
| **RNA-seq** | Bulk and single-cell transcript quantification across tissues and developmental stages |
| **Affymetrix** | Microarray hybridisation signal intensities |
| **In situ hybridisation** | Spatial expression patterns from curated literature |
| **EST** | Expressed sequence tags indicating presence of transcription |

Expression calls are summarised as **present** or **absent** per gene × anatomical entity × developmental stage, propagated through the anatomical ontology (Uberon) and ranked by expression level. The `expression_score` is a scaled rank-based measure that allows cross-tissue and cross-species comparisons.

**Key fields in expression calls:**
| Field | Description |
|---|---|
| `gene_id` | Ensembl gene identifier (e.g. `ENSG00000141510`) |
| `anatomical_entity_id` | Uberon or CL ontology term (e.g. `UBERON:0000955`) |
| `expression_score` | Rank-normalised expression level (0–100; higher = more expressed) |
| `expression_rank` | Raw rank within the species/tissue comparison |
| `call_type` | Summarised call: `EXPRESSED` or `NOT_EXPRESSED` |

**API base URL:** `https://www.bgee.org/API/`  
**Documentation:** https://www.bgee.org/support/api-documentation

**Reference:** Bastian et al. (2021), *Nucleic Acids Research*, Bgee 15: Accessing gene expression data across animal species, PMID 34912388.

# TODO

* [x] **Ingest data**
    * [x] Connect to Bgee REST API and fetch the list of available species
    * [x] Fetch gene expression calls for a well-known human gene (TP53, ENSG00000141510)
    * [x] Parse expression calls into a Polars DataFrame with correct column types
    * [x] Fetch the equivalent expression data for mouse (*Mus musculus*) TP53 orthologue
    * [x] Print shape, dtypes, and head for both DataFrames
* [ ] **Explore and clean**
    * [ ] Summarise expression call distributions (present vs. absent) per species
    * [ ] Inspect expression score distributions across anatomical entities
    * [ ] Identify top-expressing tissues for TP53 in human and mouse
    * [ ] Check for and handle missing anatomical entity names or null scores
* [ ] **Analysis**
    * [ ] Compare human vs. mouse TP53 expression profiles across shared anatomical entities
    * [ ] Compute Spearman rank correlation of expression scores between species (orthologue conservation)
    * [ ] Identify tissues where expression is conserved vs. species-specific
    * [ ] Run a TopAnat-style enrichment analysis on a gene set of interest
* [ ] **Visualization**
    * [ ] Bar chart of top-20 expressing tissues for TP53 in human
    * [ ] Side-by-side comparison of human vs. mouse expression scores in shared tissues
    * [ ] Heatmap of expression scores across tissues for a gene set
    * [ ] UMAP or hierarchical clustering of tissue expression profiles
* [ ] **Statistical analysis**
    * [ ] Explain the rank-based expression scoring model and its error properties
    * [ ] Test whether TP53 expression levels differ significantly across tissue categories
    * [ ] Apply Benjamini-Hochberg correction when comparing expression across many tissues
    * [ ] Discuss propagation of uncertainty through anatomical ontology hierarchies

In [ ]:
import json
import time
from pathlib import Path

import requests
import polars as pl

## 1. Ingest Data

### 1.1 Connect to Bgee API and Fetch Available Species

In [ ]:
BGEE_BASE = "https://www.bgee.org/API"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)  # create cache directory if it doesn't exist


def bgee_get(endpoint: str, params: dict | None = None) -> dict:
    """
    Send a GET request to the Bgee REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to BGEE_BASE (e.g. ``"species"`` or
        ``"gene/ENSG00000141510/expression"``).  A leading slash is optional.
    params : dict or None, optional
        Query parameters appended to the URL.  ``None`` sends no parameters.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a 4xx or 5xx status code.

    Notes
    -----
    A 1-second sleep after every request follows the Bgee fair-use guidance
    for programmatic access to avoid overwhelming the server.
    """
    url = f"{BGEE_BASE}/{endpoint.lstrip('/')}"
    resp = requests.get(url, params=params or {}, timeout=60)
    resp.raise_for_status()
    time.sleep(1)  # polite rate-limiting between API calls
    return resp.json()


# --- Fetch and cache the full species list ---
SPECIES_CACHE = DATA_DIR / "bgee_species.json"

if SPECIES_CACHE.exists():
    # Load from disk to avoid redundant network calls
    species_raw = json.loads(SPECIES_CACHE.read_text())
    print(f"Loaded species list from cache: {SPECIES_CACHE}")
else:
    print("Fetching species list from Bgee API …")
    species_raw = bgee_get("species")  # returns a list of species dicts
    SPECIES_CACHE.write_text(json.dumps(species_raw))
    print(f"Saved to {SPECIES_CACHE}")

# Parse into a Polars DataFrame for easy exploration
# Each record contains NCBI taxon ID, genus, species, common name, genome assembly
species_df = pl.DataFrame(
    [
        {
            "species_id":   s.get("id"),            # NCBI taxon ID (integer)
            "genus":        s.get("genus"),          # e.g. "Homo"
            "species":      s.get("speciesName"),    # e.g. "sapiens"
            "common_name":  s.get("name"),           # e.g. "human"
            "genome_version": s.get("genomeVersion"), # genome assembly used
        }
        for s in (species_raw.get("data", species_raw) if isinstance(species_raw, dict) else species_raw)
    ]
)

print(f"\nBgee covers {len(species_df)} species")
print(species_df)

### 1.2 Fetch Human TP53 Gene Expression Calls

In [ ]:
# TP53 is one of the most studied tumour suppressor genes; its broad expression
# across tissues makes it a good demonstration target.
HUMAN_TP53_ID = "ENSG00000141510"   # Ensembl stable gene ID for human TP53
HUMAN_TP53_CACHE = DATA_DIR / f"bgee_expression_human_{HUMAN_TP53_ID}.json"

if HUMAN_TP53_CACHE.exists():
    human_raw = json.loads(HUMAN_TP53_CACHE.read_text())
    print(f"Loaded from cache: {HUMAN_TP53_CACHE}")
else:
    print(f"Fetching expression data for {HUMAN_TP53_ID} (human TP53) …")
    human_raw = bgee_get(
        f"gene/{HUMAN_TP53_ID}/expression",
        params={"species_id": 9606},  # NCBI taxon ID for Homo sapiens
    )
    HUMAN_TP53_CACHE.write_text(json.dumps(human_raw))
    print(f"Saved to {HUMAN_TP53_CACHE}")

print(f"Top-level keys in response: {list(human_raw.keys()) if isinstance(human_raw, dict) else type(human_raw)}")

### 1.3 Parse Human Expression Calls into a DataFrame

In [ ]:
def parse_expression_calls(raw: dict | list, gene_id: str) -> pl.DataFrame:
    """
    Parse a Bgee gene expression API response into a tidy Polars DataFrame.

    The Bgee API wraps expression calls inside a nested JSON structure.  Each
    call represents a (gene × anatomical_entity × developmental_stage) triple
    summarised as either present (``EXPRESSED``) or absent (``NOT_EXPRESSED``).

    Parameters
    ----------
    raw : dict or list
        Parsed JSON body returned by ``bgee_get("gene/{geneId}/expression")``.
        The function handles both the ``{"data": [...]}`` envelope form and a
        bare list of call objects.
    gene_id : str
        Ensembl gene identifier to annotate the ``gene_id`` column (e.g.
        ``"ENSG00000141510"``).  Used when the response body omits it.

    Returns
    -------
    pl.DataFrame
        One row per (gene × anatomical_entity × developmental_stage) call with
        columns:

        - ``gene_id`` (``Utf8``) — Ensembl gene identifier
        - ``gene_name`` (``Utf8``) — HGNC/official gene symbol
        - ``anatomical_entity_id`` (``Utf8``) — Uberon / CL ontology term ID
        - ``anatomical_entity_name`` (``Utf8``) — human-readable tissue name
        - ``expression_score`` (``Float64``) — rank-normalised score, 0–100
        - ``expression_rank`` (``Float64``) — raw expression rank within species
        - ``call_type`` (``Utf8``) — ``"EXPRESSED"`` or ``"NOT_EXPRESSED"``

    Notes
    -----
    ``expression_score`` in Bgee is defined as
    ``(max_rank - rank) / (max_rank - min_rank) × 100``, so a score of 100
    indicates the highest observed expression level and 0 the lowest.  Ranks
    are computed within each species × condition combination after TMM
    normalisation for RNA-seq data.
    """
    # Unwrap the data envelope if present
    if isinstance(raw, dict):
        # API may nest calls under "data" -> "calls" or directly under "calls"
        calls = (
            raw.get("data", {}).get("expressionCalls", [])
            or raw.get("expressionCalls", [])
            or raw.get("data", [])
            or raw.get("calls", [])
        )
    else:
        calls = raw  # bare list

    rows = []
    for call in calls:
        # Gene information — may be nested under a "gene" sub-object
        gene_info = call.get("gene", {})
        g_id   = gene_info.get("geneId", gene_id)         # fall back to caller-supplied ID
        g_name = gene_info.get("geneName") or gene_info.get("name", "")

        # Anatomical entity — may live under "condition" -> "anatomicalEntity"
        cond   = call.get("condition", {})
        anat   = cond.get("anatomicalEntity", cond)       # some versions flatten this
        anat_id   = anat.get("id", anat.get("anatEntityId", ""))
        anat_name = anat.get("name", anat.get("anatEntityName", ""))

        # Expression score and rank
        score = call.get("expressionScore") or call.get("score")
        rank  = call.get("expressionRank")  or call.get("rank")

        # Call type: EXPRESSED or NOT_EXPRESSED
        call_type = call.get("expressionState") or call.get("callType", "")

        rows.append({
            "gene_id":               g_id,
            "gene_name":             g_name,
            "anatomical_entity_id":  anat_id,
            "anatomical_entity_name": anat_name,
            "expression_score":      float(score) if score is not None else None,
            "expression_rank":       float(rank)  if rank  is not None else None,
            "call_type":             str(call_type),
        })

    df = pl.DataFrame(rows, schema={
        "gene_id":               pl.Utf8,
        "gene_name":             pl.Utf8,
        "anatomical_entity_id":  pl.Utf8,
        "anatomical_entity_name": pl.Utf8,
        "expression_score":      pl.Float64,
        "expression_rank":       pl.Float64,
        "call_type":             pl.Utf8,
    })

    return df


human_expr = parse_expression_calls(human_raw, HUMAN_TP53_ID)

print("=== Human TP53 expression calls ===")
print(f"Shape  : {human_expr.shape}")
print(f"Dtypes : {human_expr.dtypes}")
print()
human_expr.head(10)

### 1.4 Fetch Mouse TP53 Expression Data for Cross-Species Comparison

In [ ]:
# Mouse TP53 orthologue — Ensembl ID for Mus musculus Trp53
# NCBI taxon ID 10090 = Mus musculus
MOUSE_TP53_ID = "ENSMUSG00000059552"   # Ensembl stable ID for mouse Trp53
MOUSE_TP53_CACHE = DATA_DIR / f"bgee_expression_mouse_{MOUSE_TP53_ID}.json"

if MOUSE_TP53_CACHE.exists():
    mouse_raw = json.loads(MOUSE_TP53_CACHE.read_text())
    print(f"Loaded from cache: {MOUSE_TP53_CACHE}")
else:
    print(f"Fetching expression data for {MOUSE_TP53_ID} (mouse Trp53) …")
    mouse_raw = bgee_get(
        f"gene/{MOUSE_TP53_ID}/expression",
        params={"species_id": 10090},  # NCBI taxon ID for Mus musculus
    )
    MOUSE_TP53_CACHE.write_text(json.dumps(mouse_raw))
    print(f"Saved to {MOUSE_TP53_CACHE}")

mouse_expr = parse_expression_calls(mouse_raw, MOUSE_TP53_ID)

print("=== Mouse Trp53 expression calls ===")
print(f"Shape  : {mouse_expr.shape}")
print(f"Dtypes : {mouse_expr.dtypes}")
print()
mouse_expr.head(10)

### 1.5 Summary: Shape, Dtypes, and Head for Both DataFrames

In [ ]:
# ── Side-by-side summary of both DataFrames ───────────────────────────────────
for label, df in [("Human TP53 (ENSG00000141510)", human_expr),
                  ("Mouse Trp53 (ENSMUSG00000059552)", mouse_expr)]:
    print(f"{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Shape : {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Memory: {df.estimated_size('kb'):.1f} KB")
    print()
    # Column-level type summary
    print("  Columns & dtypes:")
    for col, dtype in zip(df.columns, df.dtypes):
        null_pct = df[col].null_count() / len(df) * 100
        print(f"    {col:<30} {str(dtype):<12} nulls: {null_pct:.1f}%")
    print()
    # Call type breakdown
    print("  Call type counts:")
    print(df.group_by("call_type").len().sort("call_type"))
    print()
    # Expression score summary statistics for EXPRESSED calls
    expressed = df.filter(pl.col("call_type") == "EXPRESSED")
    if len(expressed) > 0 and expressed["expression_score"].null_count() < len(expressed):
        scores = expressed["expression_score"].drop_nulls()
        print(f"  Expression score stats (EXPRESSED calls only, n={len(scores)}):")
        print(f"    min={scores.min():.2f}  median={scores.median():.2f}  "
              f"mean={scores.mean():.2f}  max={scores.max():.2f}")
    print()

# ── Head of each DataFrame ───────────────────────────────────────────────────
print("Human TP53 — first 5 rows:")
print(human_expr.head(5))
print()
print("Mouse Trp53 — first 5 rows:")
print(mouse_expr.head(5))